# 17 pamoka – Vietinių AI agentų kūrimas su Foundry Local ir Qwen

Šiame užrašų knygelėje sukursite **vietinį inžinerinį asistentą**, kuris visiškai veikia jūsų darbo stotyje. Niekada nenaudojama jokios debesijos inferencija. Asistentas gali:

1. **Kviesti įrankius** per Qwen funkcijų kvietimus per Foundry Local.
2. **Rodyti ir skaityti failus** saugiame projekto kataloge.
3. **Analizuoti kodą** paprastais rodikliais.
4. **Ieškoti dokumentacijoje** local RAG pagalba (Chroma).
5. **Naudoti vietinį MCP serverį** (jei nėra konfigūruotas, šis žingsnis praleidžiamas sklandžiai).

Agentų kodas atrodo beveik toks pats kaip debesies pamokose – tik kliento taškas perkeltas iš debesies į `localhost`.


## Nustatymas

Prieš paleisdami šį užrašų knygelę:

1. **Įdiekite Microsoft Foundry Local** (žr. [dokumentaciją](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) savo operacinei sistemai).
2. **Atsisiųskite ir paleiskite Qwen modelį:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Įdiekite žemiau nurodytas Python paketas.

Viskas veikia vietoje. Mašina su maždaug 8 GB RAM yra realistiškas minimumas.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Prisijungimas prie Foundry Local

`FoundryLocalManager` atsisiunčia modelį, jei reikia, paleidžia vietinę paslaugą ir suteikia mums **OpenAI suderinamą galinį tašką**. Tada nukreipiame standartinį OpenAI SDK į jį. API raktas yra vietinis žymeklis — jokių debesijos kredencialų nenaudojama.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Vietiniai įrankiai (smėlio dėžės tipo failų operacijos)

Mes sukuriame nedidelį pavyzdinį projektą diske, tada apibrėžiame įrankius, apribotus tam tikram projekto šaknies katalogui. Smėlio dėžės tikrinimas yra svarbus net vietoje: įrankis, kuris skaito bet kokius kelius, veikia su jūsų vartotojo leidimais ir gali pasiekti bet ką, ką galite ir jūs.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Vietinis RAG su Chroma

Mes įterpiame nedidelį dokumentacijos ištraukų rinkinį į vietinę Chroma kolekciją. Chroma veikia proceso viduje ir išsaugo vektorius diske — be serverio, be debesies. Įrankis `search_docs` surenka tinkamiausias ištraukas užklausai.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Įrankių kvietimo ciklas

Dabar registruojame įrankius pas modelį naudodami OpenAI įrankių schemą ir vykdome standartinį įrankių kvietimo ciklą — modelis prašo įrankio, mes jį vykdome vietoje, perduodame rezultatą atgal ir kartojame, kol modelis pateikia galutinį atsakymą. Qwen patikimas funkcijų kvietimas yra tai, kas leidžia tai veikti įrenginyje.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. Vietinis MCP (pasirinktinai)

MCP yra transportas, o ne debesų paslauga — MCP serveris gali veikti kaip vietinis procesas per `stdio`. Žemiau pateikta ląstelė rodo, kaip prisijungti prie vietinio MCP serverio, jei jį turite sukonfigūravę (pavyzdžiui, failų sistemos serverį). Ji sklandžiai praleidžiama, kai `LOCAL_MCP_COMMAND` nėra nustatyta, todėl užrašų knygutė vis tiek veikia nuo pradžios iki pabaigos be jos.

Saugumo pastaba: vietinis MCP serveris veikia su jūsų vartotojo teisėmis. Ribokite jį prie projekto katalogo ir patikrinkite jo išvestis prieš imdamiesi veiksmų.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Santrauka

Jūs sukūrėte inžinerijos asistentą, kuris veikia visiškai jūsų mašinoje:

- **Foundry Local** aptarnavo **Qwen** modelį už OpenAI suderinamo vartų — taigi agento kodas atitinka debesų pamokas.
- **Sandboxed įrankiai** suteikė agentui failų prieigą ir kodo analizę nepaliekant projekto katalogo.
- **Chroma** suteikė **vietinę RAG** dokumentacijai.
- **Local MCP** parodė, kaip iš naujo naudoti MCP ekosistemą neprisijungus.

Debesų interpretacija nebuvo naudojama jokiu momentu.

### Iššūkis

Išplėskite vietinį agentą, kad:

1. **Dirbtų su keliais MCP serveriais** — prisijungtų prie failų sistemos serverio ir git serverio ir leistų agentui pasirinkti tarp jų.
2. **Naudotų vietinę atmintį** — išsaugotų trumpą pokalbio istoriją diske, kad asistentas prisimintų ankstesnius etapus per užrašų knygelės paleidimus.
3. **Palaikytų vietinę kelių agentų orkestraciją** — pridėtų antrą vietinį agentą (pvz., recenzentą) ir leistų dviem bendradarbiauti užduotyje.

Kitoje pamokoje sužinosite, kaip užtikrinti diegiamų AI agentų saugumą.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
